In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root

import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))

    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.game_model_collection import GamingModelCollection
from src.model.social_media_collection import SocialMediaModelCollection
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics


CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}


print(PROJECT_ROOT)

# social media
social_paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))

scaler_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'


# gaming
model_dir = CONFIG['model_dir'] / "binary"
gaming_model_paths = list(model_dir.glob("gaming_all_*.joblib"))



c:\Users\nyuss\OneDrive\Documentos\Bars\Portfollio\Portfollio\Projects\Project-GamingToxicityDetection


In [2]:
np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

# Load Data

In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# Prepare Objects

In [5]:
# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

gaming_preprocessor.fit(train, text_column=tc)

In [6]:
# convert dota labels 
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary'
)
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary'
)


In [7]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = gaming_model_paths
)

In [8]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False)'])

In [9]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_paths,
    scaler_path=scaler_path,
    nb_tfidf_path=nb_tfidf_path,
)

In [10]:
social_media_collection.classifiers

{WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/binary/social_media_ComplementNB_model.joblib'): Pipeline(steps=[('clf', ComplementNB(alpha=1.2006196407511314))]),
 WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib'): Pipeline(steps=[('clf',
                  LinearSVC(C=0.7813882258601701, class_weight='balanced',
                            dual=False, max_iter=5000, penalty='l1',
                            random_state=42))]),
 WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/binary/social_media_LogisticRegressionCV_model.joblib'): Pipeline(steps=[('clf',
                  LogisticRegressionCV(class_weight='balanced',
                                       cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=

In [11]:
bert_binary_collection = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_binary from jforward/bert-toxicity/dota_binary_model...
Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...
Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


In [12]:
social_gaming_ensemble = Ensemble(
    model_collections=[social_media_collection, gaming_model_collection, bert_binary_collection]
)

# Run Simple Ensemble 

In [13]:
pred_train = social_gaming_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_binary'], pred_train)

Predicting with SimpleMajority...


{'CV Macro F1': 0.9026393662138283,
 'CV Weighted F1': 0.9321364523306177,
 'Accuracy': 0.9351667380415961,
 'Coverage': 1.0,
 'Precision': 0.9701598219704632,
 'FPR': 0.00720760341078453,
 'FNR': 0.24941305368602285,
 'TPR': 0.7505869463139772,
 'TNR': 0.9927923965892155}

In [14]:
pred_val = social_gaming_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_binary'], pred_val)

Predicting with SimpleMajority...


{'CV Macro F1': 0.8198054924215161,
 'CV Weighted F1': 0.8831220604428022,
 'Accuracy': 0.8893644465290806,
 'Coverage': 1.0,
 'Precision': 0.8300653594771242,
 'FPR': 0.035053554040895815,
 'FNR': 0.38299595141700404,
 'TPR': 0.617004048582996,
 'TNR': 0.9649464459591042}

# Run Weighted Majority Ensemble 

In [ ]:
weights = social_gaming_ensemble.fit_weighted_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,zW
    n_trials=500,
    random_state=CONFIG['seed']
)[0]

In [14]:
weights

array([0.05775814, 0.00236503, 0.04977899, 0.00473355, 0.10858845,
       0.02955511, 0.04662467, 0.03615981, 0.03636258, 0.07227485,
       0.01316597, 0.20768636, 0.02183354, 0.04309042, 0.0476378 ,
       0.05308232, 0.07417326, 0.09512916])

In [15]:
pred_train = social_gaming_ensemble.predict_weighted_majority(
    X=train[tc],
    weights=weights
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.9682724687801103,
 'CV Weighted F1': 0.9769971116262947,
 'Accuracy': 0.9770048597017149,
 'Coverage': 1.0,
 'Precision': 0.9525601819179801,
 'FPR': 0.014781695130592,
 'FNR': 0.04930349037408045,
 'TPR': 0.9506965096259196,
 'TNR': 0.985218304869408}

In [16]:
pred_val = social_gaming_ensemble.predict_weighted_majority(
    X=val[tc],
    weights=weights
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8460383018697265,
 'CV Weighted F1': 0.8964717382715283,
 'Accuracy': 0.8976899624765479,
 'Coverage': 1.0,
 'Precision': 0.782258064516129,
 'FPR': 0.05662497191221631,
 'FNR': 0.2669365721997301,
 'TPR': 0.7330634278002699,
 'TNR': 0.9433750280877837}

# Run Weighted Confidence Majority Ensemble 

In [17]:
res = social_gaming_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,
    n_trials=100,
    random_state=CONFIG['seed']
)

In [18]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.9670217516015636,
 'CV Weighted F1': 0.9760687605370192,
 'Accuracy': 0.9760552628149031,
 'Coverage': 1.0,
 'Precision': 0.9482059282371295,
 'FPR': 0.016223215812748906,
 'FNR': 0.04867741430583816,
 'TPR': 0.9513225856941618,
 'TNR': 0.9837767841872511}

In [19]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8444647347142373,
 'CV Weighted F1': 0.8951506046696787,
 'Accuracy': 0.8961069418386491,
 'Coverage': 1.0,
 'Precision': 0.7743400510928187,
 'FPR': 0.059546101415624296,
 'FNR': 0.263697705802969,
 'TPR': 0.7363022941970311,
 'TNR': 0.9404538985843757}

In [20]:
thresholds = np.arange(0.5, 0.9, 0.1)

res = social_gaming_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,
    n_trials=50,
    random_state=CONFIG['seed'],
    thresholds=thresholds
)

In [21]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_binary'], pred_train)

KeyboardInterrupt: 

In [ ]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8426699101229502,
 'CV Weighted F1': 0.8940397972623165,
 'Accuracy': 0.8951102251407129,
 'Coverage': np.float64(1.0),
 'Precision': 0.7734018264840182,
 'FPR': np.float64(0.05947120065912666),
 'FNR': np.float64(0.26855600539811064),
 'TPR': np.float64(0.7314439946018894),
 'TNR': np.float64(0.9405287993408733)}